### Load packages

In [16]:
import pandas as pd
import os
import numpy as np

### Define parameters and file info

In [17]:
# -----------------------------------------------------------------------
# Directory info
# -----------------------------------------------------------------------
data_directory = r"..\data"
output_directory = r"..\output"

# -----------------------------------------------------------------------
# revenue breakout data
# -----------------------------------------------------------------------
rev_file = r"output_MCN_revenue_breakout_2023_2023_20260422_152357.csv"
rev_path = os.path.join(data_directory, rev_file)
print("Revenue CSV file:", rev_path)

# -----------------------------------------------------------------------
# Output files
# -----------------------------------------------------------------------
rev_output_file = r"MCN_revenue_breakouts_2023.csv"
rev_output_path = os.path.join(output_directory, rev_output_file)
print("Revenue output file:", rev_output_path)

Revenue CSV file: ..\data\output_MCN_revenue_breakout_2023_2023_20260422_152357.csv
Revenue output file: ..\output\MCN_revenue_breakouts_2023.csv


### Load data

In [18]:
# Load revenue breakout data

rev_raw_df = pd.read_csv(rev_path)

print(rev_raw_df.info())

rev_raw_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 262155 entries, 0 to 262154
Data columns (total 19 columns):
 #   Column                                                  Non-Null Count   Dtype  
---  ------                                                  --------------   -----  
 0   Object ID                                               262155 non-null  int64  
 1   EIN                                                     262155 non-null  int64  
 2   Name                                                    262155 non-null  object 
 3   Year of Tax Period End Date                             262155 non-null  int64  
 4   Month of Tax Period End Date                            262155 non-null  object 
 5   CY Total Revenue (Part I, Line 12, CY)                  262146 non-null  float64
 6   CY Total Expenses (Part I, Line 18, CY)                 262146 non-null  float64
 7   EOY Total Assets (Part I, Line 20, EOY)                 262146 non-null  float64
 8   CY Program Service Reven

,Object ID,EIN,Name,Year of Tax Period End Date,Month of Tax Period End Date,"CY Total Revenue (Part I, Line 12, CY)","CY Total Expenses (Part I, Line 18, CY)","EOY Total Assets (Part I, Line 20, EOY)","CY Program Service Revenue Amount (Part I, Line 9, CY)","Federated Campaigns (Part VIII, Line 1a)","Membership Dues (Part VIII, Line 1b)","Fundraising Events (Part VIII, Line 1c)","Related Organizations (Part VIII, Line 1d)","Government Grants (Part VIII, Line 1e)","All Other Contributions (Part VIII, Line 1f)","Noncash Contributions (Part VIII, Line 1g)","Total Contributions (Part VIII, Line 1h)",BMF Subsection Code,NTEE_COMBINED
0,202421369349305437,10024645,BANGOR SYMPHONY ORCHESTRA,2023,June,866598.0,1004403.0,3546551.0,376013.0,NaN,NaN,NaN,NaN,NaN,432835.0,NaN,432835.0,3.0,A69Z
1,202401299349304210,10085716,HANCOCK COUNTY AGRICULTURAL SOCIETY,2023,December,550252.0,517796.0,491794.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,N52
2,202431319349304643,10130427,BRIDGTON HOSPITAL,2023,June,70381096.0,54554893.0,38216754.0,63275237.0,NaN,NaN,24166.0,NaN,2755788.0,4066708.0,NaN,6846662.0,3.0,E220
3,202403179349306720,10133442,OXFORD COUNTY AGRICULTURAL SOCIETY,2023,December,374620.0,411510.0,415648.0,191685.0,NaN,NaN,NaN,NaN,NaN,116935.0,NaN,116935.0,3.0,Z99
4,202423059349300717,10145133,PROUTS NECK ASSOCIATION,2023,December,297189.0,151135.0,1194545.0,4400.0,NaN,NaN,NaN,NaN,NaN,265711.0,NaN,265711.0,3.0,S22


### Clean and format columns

In [19]:
rev_raw_df.columns

Index(['Object ID', 'EIN', 'Name', 'Year of Tax Period End Date',
       'Month of Tax Period End Date',
       'CY Total Revenue (Part I, Line 12, CY)',
       'CY Total Expenses (Part I, Line 18, CY)',
       'EOY Total Assets (Part I, Line 20, EOY)',
       'CY Program Service Revenue Amount (Part I, Line 9, CY)',
       'Federated Campaigns (Part VIII, Line 1a)',
       'Membership Dues (Part VIII, Line 1b)',
       'Fundraising Events (Part VIII, Line 1c)',
       'Related Organizations (Part VIII, Line 1d)',
       'Government Grants (Part VIII, Line 1e)',
       'All Other Contributions (Part VIII, Line 1f)',
       'Noncash Contributions (Part VIII, Line 1g)',
       'Total Contributions (Part VIII, Line 1h)', 'BMF Subsection Code',
       'NTEE_COMBINED'],
      dtype='object')

In [20]:
# Clean & format columns in revenue data

rev_clean_df = rev_raw_df.copy()

# Uppercase NTEE_COMBINED
rev_clean_df["NTEE_COMBINED"] = rev_clean_df["NTEE_COMBINED"].str.upper()

# Create one-digit NTEE code columm
rev_clean_df["NTEE_FIRST_DIGIT"] = rev_clean_df["NTEE_COMBINED"].str[0]

# Identify invalid first digits (not A-Z or null/NaN) and replace with "0"
rev_clean_df["NTEE_FIRST_DIGIT"] = rev_clean_df["NTEE_FIRST_DIGIT"].mask(
    rev_clean_df["NTEE_FIRST_DIGIT"].isna() | ~rev_clean_df["NTEE_FIRST_DIGIT"].str.match(r"^[A-Z]$", na=False),
    "0"
)

# Convert all float64 columns so NaNs become zeros
rev_clean_df[rev_clean_df.select_dtypes(include="float64").columns] = (
    rev_clean_df.select_dtypes(include="float64").fillna(0)
)

# Create new column to hold 1h - 1a - 1e
# Total contributions minus federated campaigns and government grants
rev_clean_df["Non-govt non-federated contributions (Line 1h minus Line 1a & Line 1e)"] = (
    rev_clean_df["Total Contributions (Part VIII, Line 1h)"] - 
    rev_clean_df["Federated Campaigns (Part VIII, Line 1a)"] - 
    rev_clean_df["Government Grants (Part VIII, Line 1e)"]
)

rev_clean_df.head()

,Object ID,EIN,Name,Year of Tax Period End Date,Month of Tax Period End Date,"CY Total Revenue (Part I, Line 12, CY)","CY Total Expenses (Part I, Line 18, CY)","EOY Total Assets (Part I, Line 20, EOY)","CY Program Service Revenue Amount (Part I, Line 9, CY)","Federated Campaigns (Part VIII, Line 1a)",...,"Fundraising Events (Part VIII, Line 1c)","Related Organizations (Part VIII, Line 1d)","Government Grants (Part VIII, Line 1e)","All Other Contributions (Part VIII, Line 1f)","Noncash Contributions (Part VIII, Line 1g)","Total Contributions (Part VIII, Line 1h)",BMF Subsection Code,NTEE_COMBINED,NTEE_FIRST_DIGIT,Non-govt non-federated contributions (Line 1h minus Line 1a & Line 1e)
0,202421369349305437,10024645,BANGOR SYMPHONY ORCHESTRA,2023,June,866598.0,1004403.0,3546551.0,376013.0,0.0,...,0.0,0.0,0.0,432835.0,0.0,432835.0,3.0,A69Z,A,432835.0
1,202401299349304210,10085716,HANCOCK COUNTY AGRICULTURAL SOCIETY,2023,December,550252.0,517796.0,491794.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,3.0,N52,N,0.0
2,202431319349304643,10130427,BRIDGTON HOSPITAL,2023,June,70381096.0,54554893.0,38216754.0,63275237.0,0.0,...,24166.0,0.0,2755788.0,4066708.0,0.0,6846662.0,3.0,E220,E,4090874.0
3,202403179349306720,10133442,OXFORD COUNTY AGRICULTURAL SOCIETY,2023,December,374620.0,411510.0,415648.0,191685.0,0.0,...,0.0,0.0,0.0,116935.0,0.0,116935.0,3.0,Z99,Z,116935.0
4,202423059349300717,10145133,PROUTS NECK ASSOCIATION,2023,December,297189.0,151135.0,1194545.0,4400.0,0.0,...,0.0,0.0,0.0,265711.0,0.0,265711.0,3.0,S22,S,265711.0


### Output results

In [21]:
# Output the alldaf grants files

rev_output_df = rev_clean_df.copy()
rev_output_df.to_csv(rev_output_path, index=False)